# DF40 Dataset — Split Vidéo-Level 20/40/40 (Notebook 03)


# CELLULE 1 — MONTAGE DRIVE ET CONFIGURATION

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install pandas numpy tqdm -q

import os
import random
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from tqdm import tqdm

# ── Seed de reproductibilité ──────────────────────────────────────────────────
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ── Chemins ───────────────────────────────────────────────────────────────────
PROJECT_ROOT  = '/content/drive/MyDrive/Memoire_Deepfakes'
DATA_DIR      = f'{PROJECT_ROOT}/data'
RAW_DIR       = f'{DATA_DIR}/raw'
REAL_DIR      = f'{RAW_DIR}/DF40_real'
FAKE_BASE_DIR = f'{RAW_DIR}/DF40_fake_DMs'
MANIFEST_PATH = f'{DATA_DIR}/real_video_manifest.csv'
SPLITS_DIR    = f'{DATA_DIR}/splits'
os.makedirs(SPLITS_DIR, exist_ok=True)

# ── Paramètres du split ───────────────────────────────────────────────────────
SPLIT_RATIOS = {'train': 0.20, 'test': 0.40, 'val': 0.40}
DM_METHODS   = ['MidJourney', 'ddim', 'DiT', 'SiT']

# ── Chiffres attendus post-audit ──────────────────────────────────────────────
EXPECTED = {
    'REAL_TOTAL' : 3_510,
    'FAKE_TOTAL' : 3_510,
    'GRAND_TOTAL': 7_020,
}

def count_images(directory):
    if not os.path.isdir(directory):
        return 0
    return len([f for f in os.listdir(directory)
                if Path(f).suffix.lower() in {'.jpg', '.jpeg', '.png'}])

# ── Vérification de l'état du dataset ────────────────────────────────────────
real_count = count_images(REAL_DIR)
fake_counts = {m: count_images(os.path.join(FAKE_BASE_DIR, m)) for m in DM_METHODS}
fake_total  = sum(fake_counts.values())
grand_total = real_count + fake_total

print('=' * 65)
print('NOTEBOOK 03 — SPLIT VIDÉO-LEVEL 20/40/40')
print('=' * 65)
print(f'\n  État du dataset (pré-split) :')
print(f'  REAL  : {real_count:,}  {"✅" if real_count == EXPECTED["REAL_TOTAL"] else "⚠️"}')
for m in DM_METHODS:
    print(f'  {m:<15s}: {fake_counts[m]:,}')
print(f'  FAKE  : {fake_total:,}  {"✅" if fake_total == EXPECTED["FAKE_TOTAL"] else "⚠️"}')
print(f'  TOTAL : {grand_total:,}  {"✅" if grand_total == EXPECTED["GRAND_TOTAL"] else "⚠️"}')

assert grand_total == EXPECTED['GRAND_TOTAL'], \
    f'Dataset non conforme : {grand_total} ≠ {EXPECTED["GRAND_TOTAL"]}. Vérifier notebook 02b.'

print('\n  ✅ Dataset conforme — démarrage du split')
print('=' * 65)


# CELLULE 2 — SPLIT REAL (vidéo-level)

Lecture de real_video_manifest.csv.  
Construction des clés vidéo uniques prefix_videoId.  
Shuffle avec seed=42, puis découpe 20/40/40 au niveau vidéo.  
Toutes les frames d'une même vidéo vont dans le même set.


In [ ]:
print('=' * 65)
print('SPLIT REAL — VIDÉO-LEVEL')
print('=' * 65)

# ── Chargement du manifeste ───────────────────────────────────────────────────
manifest = pd.read_csv(MANIFEST_PATH)
print(f'  Manifeste : {len(manifest):,} lignes')

# Clé unique par vidéo (video_id est un entier qui repart de 0 par source)
manifest['unique_vid'] = (
    manifest['prefix'].astype(str) + '_' + manifest['video_id'].astype(str)
)

# ── Liste des vidéos uniques et shuffle reproductible ────────────────────────
all_vids = manifest['unique_vid'].unique().tolist()
n_vids   = len(all_vids)
random.seed(RANDOM_SEED)
random.shuffle(all_vids)
print(f'  Vidéos uniques : {n_vids:,}')

# ── Calcul des seuils (floor pour train, floor pour test, reste pour val) ─────
n_train_vids = int(np.floor(n_vids * SPLIT_RATIOS['train']))
n_test_vids  = int(np.floor(n_vids * SPLIT_RATIOS['test']))
n_val_vids   = n_vids - n_train_vids - n_test_vids

train_vids = set(all_vids[:n_train_vids])
test_vids  = set(all_vids[n_train_vids : n_train_vids + n_test_vids])
val_vids   = set(all_vids[n_train_vids + n_test_vids:])

assert len(train_vids) + len(test_vids) + len(val_vids) == n_vids, 'Partition incomplète'
assert len(train_vids & test_vids) == 0, 'Overlap train/test'
assert len(train_vids & val_vids)  == 0, 'Overlap train/val'
assert len(test_vids  & val_vids)  == 0, 'Overlap test/val'

# ── Assignation des sets dans le manifeste ────────────────────────────────────
def assign_split_real(uid):
    if uid in train_vids: return 'train'
    if uid in test_vids:  return 'test'
    return 'val'

manifest['split'] = manifest['unique_vid'].map(assign_split_real)

# ── Statistiques par split ────────────────────────────────────────────────────
real_split_counts = manifest.groupby('split').size()
real_source_split = manifest.groupby(['source', 'split']).size().unstack(fill_value=0)

print(f'\n  Vidéos par split :')
print(f'    train : {n_train_vids:>5} vidéos → {real_split_counts.get("train", 0):>5} images ({n_train_vids/n_vids*100:.1f}%)')
print(f'    test  : {n_test_vids:>5} vidéos → {real_split_counts.get("test", 0):>5} images ({n_test_vids/n_vids*100:.1f}%)')
print(f'    val   : {n_val_vids:>5} vidéos → {real_split_counts.get("val", 0):>5} images ({n_val_vids/n_vids*100:.1f}%)')

print(f'\n  Distribution par source :')
print(real_source_split.to_string())

# ── Préparation du DataFrame REAL pour le manifeste final ────────────────────
real_records = []
for _, row in manifest.iterrows():
    real_records.append({
        'filename' : row['filename'],
        'filepath' : os.path.join(REAL_DIR, row['filename']),
        'label'    : 0,           # 0 = REAL
        'source'   : row['source'],
        'method'   : 'REAL',
        'split'    : row['split'],
    })

df_real = pd.DataFrame(real_records)
print(f'\n  ✅ REAL splitté : {len(df_real):,} entrées')
print('  ✅ Aucun overlap vidéo inter-sets — data leakage ÉLIMINÉ')


# CELLULE 3 — SPLIT FAKE

Chaque méthode DM est splitée indépendamment avec les ratios 20/40/40.  

In [ ]:
print('=' * 65)
print('SPLIT FAKE — STRATIFIÉ PAR MÉTHODE DM')
print('=' * 65)

fake_records = []
fake_split_summary = {}

random.seed(RANDOM_SEED)   # Reset seed pour reproductibilité

for method in DM_METHODS:
    method_dir = os.path.join(FAKE_BASE_DIR, method)
    files = sorted([
        f for f in os.listdir(method_dir)
        if Path(f).suffix.lower() in {'.jpg', '.jpeg', '.png'}
    ])
    n = len(files)

    # Shuffle reproductible (seed différent par méthode via random state continu)
    random.shuffle(files)

    # Seuils : floor pour train et test, reste pour val
    n_train = int(np.floor(n * SPLIT_RATIOS['train']))
    n_test  = int(np.floor(n * SPLIT_RATIOS['test']))
    n_val   = n - n_train - n_test

    splits_for_method = (
        [('train', f) for f in files[:n_train]] +
        [('test',  f) for f in files[n_train : n_train + n_test]] +
        [('val',   f) for f in files[n_train + n_test:]]
    )

    for split_name, fname in splits_for_method:
        fake_records.append({
            'filename' : fname,
            'filepath' : os.path.join(method_dir, fname),
            'label'    : 1,           # 1 = FAKE
            'source'   : 'DF40_DM',
            'method'   : method,
            'split'    : split_name,
        })

    fake_split_summary[method] = {
        'total': n, 'train': n_train, 'test': n_test, 'val': n_val
    }

    pct_train = n_train / n * 100
    pct_test  = n_test  / n * 100
    pct_val   = n_val   / n * 100
    print(f'  {method:<15s} (n={n:,}) : '
          f'train={n_train} ({pct_train:.1f}%) | '
          f'test={n_test} ({pct_test:.1f}%) | '
          f'val={n_val} ({pct_val:.1f}%)')

df_fake = pd.DataFrame(fake_records)
print(f'\n  ✅ FAKE splitté : {len(df_fake):,} entrées')


# CELLULE 4 — ASSEMBLAGE ET VALIDATION DU MANIFESTE FINAL

Fusion REAL + FAKE, vérifications d'intégrité, et validation des ratios par set.

In [ ]:
print('=' * 65)
print('ASSEMBLAGE ET VALIDATION DU MANIFESTE FINAL')
print('=' * 65)

# ── Fusion REAL + FAKE ────────────────────────────────────────────────────────
df_all = pd.concat([df_real, df_fake], ignore_index=True)

# ── Vérifications d'intégrité ─────────────────────────────────────────────────
assert len(df_all) == EXPECTED['GRAND_TOTAL'], \
    f'Total inattendu : {len(df_all)} ≠ {EXPECTED["GRAND_TOTAL"]}'
assert df_all['split'].isin(['train', 'test', 'val']).all(), \
    'Certaines lignes n\'ont pas de split assigné'
assert df_all['filename'].nunique() == len(df_all), \
    f'Doublons détectés dans les noms de fichiers'
assert set(df_all['label'].unique()) == {0, 1}, \
    'Labels inattendus'

# ── Statistiques globales par split ──────────────────────────────────────────
split_stats = df_all.groupby(['split', 'label']).size().unstack(fill_value=0)
split_stats.columns = ['REAL', 'FAKE']
split_stats['TOTAL'] = split_stats['REAL'] + split_stats['FAKE']
split_stats['Ratio R/F'] = (split_stats['REAL'] / split_stats['FAKE']).round(4)
split_stats['% du total'] = (split_stats['TOTAL'] / len(df_all) * 100).round(1)

# Reorder
split_stats = split_stats.loc[['train', 'test', 'val']]

print(f'\n  Manifeste complet : {len(df_all):,} images\n')
print(f'  {"Set":<8} {"REAL":>6} {"FAKE":>6} {"TOTAL":>7} {"Ratio R/F":>10} {"% total":>8}')
print('  ' + '-' * 48)
for split_name, row in split_stats.iterrows():
    ratio_flag = '✅' if abs(row['Ratio R/F'] - 1.0) < 0.05 else '⚠️'
    print(f'  {split_name:<8} {int(row["REAL"]):>6,} {int(row["FAKE"]):>6,} '
          f'{int(row["TOTAL"]):>7,} {row["Ratio R/F"]:>10.4f} {row["% du total"]:>7.1f}%  {ratio_flag}')
print('  ' + '-' * 48)
print(f'  {"TOTAL":<8} {int(split_stats["REAL"].sum()):>6,} '
      f'{int(split_stats["FAKE"].sum()):>6,} '
      f'{int(split_stats["TOTAL"].sum()):>7,}')

# ── Distribution des méthodes DM par split ───────────────────────────────────
print(f'\n  Distribution méthodes FAKE par split :')
dm_dist = df_fake.groupby(['method', 'split']).size().unstack(fill_value=0)
dm_dist = dm_dist[['train', 'test', 'val']]
print(dm_dist.to_string())

print('\n  ✅ Toutes les vérifications passées')


# CELLULE 5 — SAUVEGARDE DES MANIFESTES CSV

Génère 4 fichiers dans data/splits/

⚠️ **`val_manifest.csv` est le coffre-fort.** Ne pas l'utiliser avant l'évaluation finale.


In [ ]:
print('=' * 65)
print('SAUVEGARDE DES MANIFESTES CSV')
print('=' * 65)

# ── Manifeste complet ─────────────────────────────────────────────────────────
master_path = f'{SPLITS_DIR}/split_manifest.csv'
df_all.to_csv(master_path, index=False)
print(f'  ✅ split_manifest.csv    : {len(df_all):,} lignes')

# ── Manifestes par set ────────────────────────────────────────────────────────
for split_name in ['train', 'test', 'val']:
    df_split = df_all[df_all['split'] == split_name].copy()
    out_path = f'{SPLITS_DIR}/{split_name}_manifest.csv'
    df_split.to_csv(out_path, index=False)
    size_kb = os.path.getsize(out_path) / 1024
    vault_tag = '  ⚠️  COFFRE-FORT — ne pas ouvrir avant évaluation finale' if split_name == 'val' else ''
    print(f'  ✅ {split_name}_manifest.csv   : {len(df_split):,} lignes  ({size_kb:.1f} KB){vault_tag}')

# ── Rapport formel ────────────────────────────────────────────────────────────
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
report_path = f'{SPLITS_DIR}/split_report_{ts}.txt'

with open(report_path, 'w', encoding='utf-8') as f:
    sep = '=' * 65
    f.write(sep + '\n')
    f.write('RAPPORT DE SPLIT — DATASET DF40 v4\n')
    f.write('Mémoire : Obsolescence des détecteurs de deepfakes\n')
    f.write('Auteur  : Maxime Ducarme\n')
    f.write(sep + '\n')
    f.write(f'Date    : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')
    f.write(f'Seed    : {RANDOM_SEED}\n')
    f.write(f'Total   : {len(df_all):,} images\n\n')

    f.write('PARAMÈTRES\n')
    f.write('-' * 65 + '\n')
    f.write(f'  Ratios    : train={SPLIT_RATIOS["train"]} / test={SPLIT_RATIOS["test"]} / val={SPLIT_RATIOS["val"]}\n')
    f.write(f'  REAL      : split vidéo-level (anti data-leakage identité)\n')
    f.write(f'  FAKE      : split stratifié par méthode DM\n\n')

    f.write('DISTRIBUTION PAR SET\n')
    f.write('-' * 65 + '\n')
    for split_name in ['train', 'test', 'val']:
        df_s = df_all[df_all['split'] == split_name]
        n_real = (df_s['label'] == 0).sum()
        n_fake = (df_s['label'] == 1).sum()
        f.write(f'  {split_name:<6} : {len(df_s):,} images '
                f'(REAL={n_real:,}, FAKE={n_fake:,}, '
                f'ratio={n_real/max(n_fake,1):.4f})\n')

    f.write('\nDISTRIBUTION FAKE PAR MÉTHODE ET SET\n')
    f.write('-' * 65 + '\n')
    for method in DM_METHODS:
        s = fake_split_summary[method]
        f.write(f'  {method:<15s}: total={s["total"]:,} | '
                f'train={s["train"]:,} | test={s["test"]:,} | val={s["val"]:,}\n')

    f.write('\nDISTRIBUTION REAL PAR SOURCE ET SET\n')
    f.write('-' * 65 + '\n')
    for src in manifest['source'].unique():
        src_data = manifest[manifest['source'] == src]
        for split_name in ['train', 'test', 'val']:
            cnt = (src_data['split'] == split_name).sum()
            f.write(f'  {src:<20s} {split_name:<6}: {cnt:,} images\n')

    f.write('\nFICHIERS GÉNÉRÉS\n')
    f.write('-' * 65 + '\n')
    for fname in ['split_manifest.csv', 'train_manifest.csv',
                  'test_manifest.csv', 'val_manifest.csv']:
        p = f'{SPLITS_DIR}/{fname}'
        sz = os.path.getsize(p) / 1024 if os.path.exists(p) else 0
        f.write(f'  {fname:<30s}: {sz:.1f} KB\n')
    f.write(sep + '\n')

print(f'\n  ✅ Rapport sauvegardé : {report_path}')


# CELLULE 6 — RÉSUMÉ FINAL ET VÉRIFICATION DES FICHIERS

In [ ]:
print('=' * 65)
print('NOTEBOOK 03 — RÉSUMÉ FINAL')
print('=' * 65)

# Vérification de cohérence finale : rechargement depuis disque
df_check = pd.read_csv(master_path)
assert len(df_check) == EXPECTED['GRAND_TOTAL'], 'Erreur lecture manifeste'
assert df_check['filename'].nunique() == len(df_check), 'Doublons dans manifeste'

print(f'\n  Manifeste relu depuis disque : {len(df_check):,} lignes  ✅')
print(f'  Fichiers uniques             : {df_check["filename"].nunique():,}  ✅')

print(f'\n  Colonnes disponibles pour le notebook 04 :')
for col in df_check.columns:
    sample = df_check[col].iloc[0]
    print(f'    {col:<12s} : {sample}')

print(f'\n  Fichiers dans data/splits/ :')
for fname in sorted(os.listdir(SPLITS_DIR)):
    fpath = os.path.join(SPLITS_DIR, fname)
    sz = os.path.getsize(fpath) / 1024
    vault = '  ⚠️  COFFRE-FORT' if fname == 'val_manifest.csv' else ''
    print(f'    {fname:<35s} {sz:>8.1f} KB{vault}')

print()
print('  ✅ SPLIT TERMINÉ AVEC SUCCÈS')
print()
print('  RÈGLES D\'USAGE :')
print('    [train] → méta-learner (Scénario C) uniquement')
print('    [test]  → évaluation Scénarios A, B, C en cours de dev')
print('    [val]   → ⚠️  COFFRE-FORT — ouverture unique notebook 06')
print()
print('  PROCHAINE ÉTAPE :')
print('    Notebook 04 — Inférence : charger train_manifest.csv et test_manifest.csv')
print('    Passer les 4 modèles (poids gelés) sur chaque image → CSV de probabilités')
print('=' * 65)
